In [3]:
import xgboost
!pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpy

In [15]:
import pandas as pd

df = pd.read_csv("flights.csv", nrows=200000)
df.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,408.0,-22.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,741.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,811.0,5.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,756.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,259.0,-21.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
df.info()
df.shape


<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 31 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   YEAR                 200000 non-null  int64  
 1   MONTH                200000 non-null  int64  
 2   DAY                  200000 non-null  int64  
 3   DAY_OF_WEEK          200000 non-null  int64  
 4   AIRLINE              200000 non-null  str    
 5   FLIGHT_NUMBER        200000 non-null  int64  
 6   TAIL_NUMBER          199598 non-null  str    
 7   ORIGIN_AIRPORT       200000 non-null  str    
 8   DESTINATION_AIRPORT  200000 non-null  str    
 9   SCHEDULED_DEPARTURE  200000 non-null  int64  
 10  DEPARTURE_TIME       195132 non-null  float64
 11  DEPARTURE_DELAY      195132 non-null  float64
 12  TAXI_OUT             194975 non-null  float64
 13  WHEELS_OFF           194975 non-null  float64
 14  SCHEDULED_TIME       200000 non-null  int64  
 15  ELAPSED_TIME         194401 

(200000, 31)

In [19]:
missing = df.isnull().sum().sort_values(ascending=False)
print(missing[missing > 0])

CANCELLATION_REASON    194923
LATE_AIRCRAFT_DELAY    140399
WEATHER_DELAY          140399
AIRLINE_DELAY          140399
AIR_SYSTEM_DELAY       140399
SECURITY_DELAY         140399
ELAPSED_TIME             5599
AIR_TIME                 5599
ARRIVAL_DELAY            5599
WHEELS_ON                5234
TAXI_IN                  5234
ARRIVAL_TIME             5234
WHEELS_OFF               5025
TAXI_OUT                 5025
DEPARTURE_TIME           4868
DEPARTURE_DELAY          4868
TAIL_NUMBER               402
dtype: int64


In [20]:
df.describe()

,YEAR,MONTH,DAY,DAY_OF_WEEK,FLIGHT_NUMBER,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,...,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
count,200000.0,200000.0,200000.000000,200000.000000,200000.000000,200000.000000,195132.000000,195132.000000,194975.000000,194975.00000,...,200000.000000,194766.000000,194401.000000,200000.000000,200000.000000,59601.000000,59601.000000,59601.000000,59601.000000,59601.000000
mean,2015.0,1.0,6.937105,4.039710,2260.089820,1328.138115,1345.905566,16.952545,16.959133,1368.56698,...,1507.263060,1489.747179,14.510841,0.002610,0.025385,13.850422,0.062851,17.760272,25.565980,3.300196
std,0.0,0.0,3.716896,2.064707,1811.454112,471.807126,489.201458,43.681561,10.268420,490.04468,...,489.080817,522.343409,46.626136,0.051022,0.157292,26.633089,1.584534,43.055289,42.862698,20.444889
min,2015.0,1.0,1.000000,1.000000,1.000000,5.000000,1.000000,-42.000000,1.000000,1.00000,...,1.000000,1.000000,-74.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2015.0,1.0,4.000000,2.000000,760.000000,927.000000,935.000000,-4.000000,11.000000,950.00000,...,1123.000000,1114.000000,-10.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2015.0,1.0,7.000000,4.000000,1720.500000,1324.000000,1338.000000,1.000000,14.000000,1351.00000,...,1526.000000,1523.000000,1.000000,0.000000,0.000000,4.000000,0.000000,3.000000,9.000000,0.000000
75%,2015.0,1.0,10.000000,6.000000,3470.000000,1725.000000,1740.000000,20.000000,20.000000,1754.00000,...,1917.000000,1921.000000,21.000000,0.000000,0.000000,18.000000,0.000000,19.000000,34.000000,0.000000
max,2015.0,1.0,13.000000,7.000000,7438.000000,2359.000000,2400.000000,1450.000000,176.000000,2400.00000,...,2359.000000,2400.000000,1444.000000,1.000000,1.000000,824.000000,107.000000,1444.000000,938.000000,938.000000


In [21]:
df["is_delayed"] = (df["ARRIVAL_DELAY"] > 15).astype(int)
df["is_delayed"].value_counts()

is_delayed
0    142245
1     57755
Name: count, dtype: int64

In [22]:
df["CANCELLED"].value_counts()


CANCELLED
0    194923
1      5077
Name: count, dtype: int64

In [24]:
df = df[df["ARRIVAL_DELAY"].notnull()]

In [25]:
missing = df.isnull().sum().sort_values(ascending=False)
print(missing[missing > 0])

CANCELLATION_REASON    194401
SECURITY_DELAY         134800
WEATHER_DELAY          134800
LATE_AIRCRAFT_DELAY    134800
AIRLINE_DELAY          134800
AIR_SYSTEM_DELAY       134800
dtype: int64


In [27]:
df["is_delayed"] = (df["ARRIVAL_DELAY"] > 15).astype(int)
df["is_delayed"].value_counts(normalize=True)

is_delayed
0    0.702908
1    0.297092
Name: proportion, dtype: float64

In [28]:
df["DIVERTED"].value_counts()

DIVERTED
0    194401
Name: count, dtype: int64

In [29]:
df["CANCELLED"].value_counts()

CANCELLED
0    194401
Name: count, dtype: int64

In [30]:
df = df.drop(columns=["CANCELLED", "DIVERTED", "CANCELLATION_REASON"])

In [32]:
df["DEPARTURE_HOUR"] = df["DEPARTURE_TIME"] // 100

In [33]:
df["DEPARTURE_HOUR"].value_counts().sort_index()

DEPARTURE_HOUR
0.0       794
1.0       384
2.0       116
3.0        51
4.0       181
5.0      5122
6.0     11953
7.0     11277
8.0     12175
9.0     11497
10.0    11979
11.0    12321
12.0    11571
13.0    12394
14.0    11757
15.0    11972
16.0    11892
17.0    12619
18.0    11075
19.0    11211
20.0     9061
21.0     6720
22.0     4303
23.0     1953
24.0       23
Name: count, dtype: int64

In [34]:
df["DEPARTURE_HOUR"]


0         23.0
1          0.0
2          0.0
3          0.0
4          0.0
          ... 
199995    22.0
199996    22.0
199997    22.0
199998    22.0
199999    22.0
Name: DEPARTURE_HOUR, Length: 194401, dtype: float64

In [47]:
def categorize_hour(h):
    if 5 <= h < 9:
        return "morning_rush"
    elif 9 <= h < 16:
        return "midday"
    elif 16 <= h < 20:
        return "evening_rush"
    else:
        return "low_traffic"

df["TIME_BLOCK"] = df["DEPARTURE_HOUR"].apply(categorize_hour)

In [48]:
df.columns

Index(['YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'FLIGHT_NUMBER',
       'TAIL_NUMBER', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT',
       'SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'DEPARTURE_DELAY', 'TAXI_OUT',
       'WHEELS_OFF', 'SCHEDULED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE',
       'WHEELS_ON', 'TAXI_IN', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME',
       'ARRIVAL_DELAY', 'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY',
       'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY', 'is_delayed', 'departure_hour',
       'DEPARTURE_HOUR', 'TIME_BLOCK_low_traffic', 'TIME_BLOCK_midday',
       'TIME_BLOCK_morning_rush', 'TIME_BLOCK'],
      dtype='str')

In [49]:
df.groupby("TIME_BLOCK")["is_delayed"].mean().sort_values(ascending=False)

TIME_BLOCK
low_traffic     0.481472
evening_rush    0.374191
midday          0.274652
morning_rush    0.146988
Name: is_delayed, dtype: float64

In [50]:
df = pd.get_dummies(df, columns=["TIME_BLOCK"], drop_first=True)

In [52]:
df = df.loc[:, ~df.columns.duplicated()]

In [53]:
df.columns

Index(['YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'FLIGHT_NUMBER',
       'TAIL_NUMBER', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT',
       'SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'DEPARTURE_DELAY', 'TAXI_OUT',
       'WHEELS_OFF', 'SCHEDULED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE',
       'WHEELS_ON', 'TAXI_IN', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME',
       'ARRIVAL_DELAY', 'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY',
       'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY', 'is_delayed', 'departure_hour',
       'DEPARTURE_HOUR', 'TIME_BLOCK_low_traffic', 'TIME_BLOCK_midday',
       'TIME_BLOCK_morning_rush'],
      dtype='str')

In [55]:
[col for col in df.columns if "TIME_BLOCK" in col]

['TIME_BLOCK_low_traffic', 'TIME_BLOCK_midday', 'TIME_BLOCK_morning_rush']

In [56]:
features = [
    "DEPARTURE_HOUR",
    "DISTANCE",
    "SCHEDULED_TIME",
    "TAXI_OUT",
    "TIME_BLOCK_low_traffic",
    "TIME_BLOCK_midday",
    "TIME_BLOCK_morning_rush"]

x = df[features]
y = df["is_delayed"]

In [57]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [62]:
from sklearn.linear_model import LogisticRegression\

model = LogisticRegression(max_iter=1000, class_weight="balanced")

model.fit(x_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [73]:
from sklearn.metrics import accuracy_score, classification_report

threshold = 0.4
threshold = 0.45
y_pred_custom = (y_probs > threshold).astype(int)
y_pred = model.predict(x_test)

print("Accuracy:", accuracy_score(y_test, y_pred_custom))
print(classification_report(y_test, y_pred_custom))

Accuracy: 0.6098865769913325
              precision    recall  f1-score   support

           0       0.83      0.56      0.67     27511
           1       0.41      0.72      0.52     11370

    accuracy                           0.61     38881
   macro avg       0.62      0.64      0.60     38881
weighted avg       0.71      0.61      0.63     38881



In [77]:
from sklearn.ensemble import RandomForestClassifier

threshold = 0.45

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(x_train, y_train)

y_probs_rf = rf.predict_proba(x_test)[:,1]
y_pred_rf = (y_probs_rf > 0.45).astype(int)

results = []

for t in [0.3, 0.35, 0.4, 0.45, 0.5]:
    y_pred = (y_probs > t).astype(int)

    from sklearn.metrics import precision_score, recall_score, f1_score

    results.append({
        "threshold": t,
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred)
    })

pd.DataFrame(results)

,threshold,precision,recall,f1
0,0.30,0.336861,0.946262,0.496848
1,0.35,0.353433,0.899033,0.507396
2,0.40,0.376469,0.816974,0.515426
3,0.45,0.406055,0.721900,0.519757
4,0.50,0.440015,0.615479,0.513163


In [78]:
!pip install xgboost


   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 1.0/101.7 MB 4.1 MB/s eta 0:00:25
    --------------------------------------- 1.8/101.7 MB 4.2 MB/s eta 0:00:24
    --------------------------------------- 2.4/101.7 MB 3.2 MB/s eta 0:00:32
   - -------------------------------------- 2.6/101.7 MB 3.0 MB/s eta 0:00:34
   - -------------------------------------- 2.9/101.7 MB 2.5 MB/s eta 0:00:41
   - -------------------------------------- 2.9/101.7 MB 2.5 MB/s eta 0:00:41
   - -------------------------------------- 3.1/101.7 MB 2.1 MB/s eta 0:00:47
   - -------------------------------------- 3.4/101.7 MB 1.9 MB/s eta 0:00:51
   - -------------------------------------- 3.7/101.7 MB 1.8 MB/s eta 0:00:55
   - -------------------------------------- 3.7/101.7 MB 1.8 MB/s eta 0:00:55
   - --

In [85]:
from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(x_train, y_train)

y_probs_xgb = xgb.predict_proba(x_test)[:, 1]

from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

result = []

for t in [0.3, 0.35, 0.4, 0.45, 0.5]:
    y_pred = (y_probs_xgb > t).astype(int)

    results.append({
        "threshold": t,
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred)
    })

    print(pd.DataFrame(results))

    from sklearn.metrics import classification_report

    best_t = 0.45
    y_pred = (y_probs_xgb > best_t).astype(int)

    print(classification_report(y_test, y_pred))

   threshold  precision    recall        f1
0       0.30   0.336861  0.946262  0.496848
1       0.35   0.353433  0.899033  0.507396
2       0.40   0.376469  0.816974  0.515426
3       0.45   0.406055  0.721900  0.519757
4       0.50   0.440015  0.615479  0.513163
5       0.30   0.351757  0.926913  0.509980
              precision    recall  f1-score   support

           0       0.85      0.58      0.69     27511
           1       0.43      0.75      0.54     11370

    accuracy                           0.63     38881
   macro avg       0.64      0.67      0.62     38881
weighted avg       0.73      0.63      0.65     38881

   threshold  precision    recall        f1
0       0.30   0.336861  0.946262  0.496848
1       0.35   0.353433  0.899033  0.507396
2       0.40   0.376469  0.816974  0.515426
3       0.45   0.406055  0.721900  0.519757
4       0.50   0.440015  0.615479  0.513163
5       0.30   0.351757  0.926913  0.509980
6       0.35   0.370282  0.888742  0.522763
             